In [1]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy.stats import mannwhitneyu

In [8]:
# Set to None to display all columns
pd.set_option('display.max_columns', None)

# To also display all rows if needed
pd.set_option('display.max_rows', None)

'''

=======================================================
SUMMARY: Consistently Wrong Antibodies
=======================================================
  antibody_name  Titer  n_models_high_err  mean_abserr   HIC   HAC  SEC %Monomer  AC-SINS_pH6.0 hc_subtype lc_subtype
   emibetuzumab 781.93                  5   509.639297 2.785   NaN        91.580          12.25       IgG4      Kappa
     inotuzumab 637.93                  5   400.418494 2.760 4.785        85.305           2.85       IgG4      Kappa
    eptinezumab 642.34                  5   360.173411 3.680   NaN        99.675          -0.14       IgG1      Kappa
      orticumab 700.57                  5   356.455963 2.520   NaN        99.795           2.60       IgG2     Lambda
    figitumumab 571.63                  5   352.723352 2.990   NaN        99.520          13.96       IgG2      Kappa
    lexatumumab  52.18                  5   321.476039 2.500   NaN        96.440           0.75       IgG1     Lambda
     olokizumab 516.74                  5   277.155538 2.840   NaN        91.080           1.35       IgG4      Kappa
    dalotuzumab 464.78                  5   271.620236 2.730   NaN        98.360           0.60       IgG1      Kappa
    epratuzumab 491.08                  5   260.903758 2.545 4.010        97.045           0.86       IgG1      Kappa
    mitazalimab 678.57                  5   259.348941 2.735 4.110        98.340          23.35       IgG1     Lambda
      tovetumab 491.52                  5   239.305996 2.500 3.880        87.910           0.61       IgG2      Kappa
    dacetuzumab  34.26                  5   227.059717 2.480   NaN        92.020           0.25       IgG1      Kappa
   farletuzumab 464.78                  5   217.756912 2.660 3.650        98.175           0.10       IgG1      Kappa
      oleclumab 591.12                  5   202.308685 2.955 4.210        98.590           3.85       IgG1     Lambda
     cemiplimab  72.14                  5   200.551201 2.730   NaN        84.655           1.86       IgG4      Kappa
  cabiralizumab  57.13                  5   183.182051 2.720   NaN        87.140           0.35       IgG4      Kappa
      icrucumab  46.59                  5   178.874155 3.280   NaN        95.980           0.75       IgG1      Kappa
    ublituximab  46.59                  5   178.385910 3.140   NaN        95.020           0.75       IgG1      Kappa
     utomilumab 438.38                  5   175.676721 3.035   NaN        99.845          16.11       IgG2     Lambda
   lebrikizumab  57.13                  5   172.619080 3.485   NaN        89.835           0.85       IgG4      Kappa
    motavizumab 383.43                  5   168.880717 2.695   NaN        97.825           0.35       IgG1      Kappa
    nimotuzumab  60.31                  5   165.564249 2.730   NaN        94.940           0.00       IgG1      Kappa
      galiximab 179.14                  5   163.836364 3.400   NaN        98.410          -0.04       IgG1     Lambda
     cetrelimab  69.28                  5   163.810662 3.400   NaN        95.640          15.75       IgG4      Kappa
      sirukumab  83.88                  5   163.497677 3.180   NaN        99.040          17.86       IgG1      Kappa
     emapalumab 202.74                  5   162.895814 2.795 1.810        94.630           0.61       IgG1     Lambda
     patritumab  66.54                  5   161.802758 2.845   NaN        94.480           1.36       IgG1      Kappa
   zolbetuximab 375.20                  5   161.309254 2.530   NaN        99.465          30.60       IgG1      Kappa
    ocrelizumab 420.17                  5   160.769076 2.860   NaN        99.615           1.61       IgG1      Kappa
    romosozumab 393.92                  5   160.428236 2.565   NaN        98.360           0.35       IgG2      Kappa
   otlertuzumab  66.54                  5   160.053292 3.080   NaN        93.990           0.25       IgG1      Kappa
    palivizumab 367.30                  5   156.258284 2.570 4.040        97.840          -0.25       IgG1      Kappa
    fezakinumab 251.19                  5   152.096022 3.305   NaN        96.035           0.36       IgG1     Lambda
    tigatuzumab 121.41                  5   145.300499 2.885   NaN        98.930           1.86       IgG1      Kappa
rozanolixizumab 125.49                  5   141.801515 2.575   NaN        87.580           0.86       IgG4      Kappa
    onartuzumab 375.20                  5   140.113463 2.760   NaN        93.020          -0.39       IgG1      Kappa
      fasinumab  88.48                  5   136.748164 2.770 1.505        88.340           0.86       IgG4      Kappa
     alirocumab  89.39                  5   136.007169 2.535   NaN        96.720          -0.14       IgG1      Kappa

=======================================================
GROUP MEDIANS (numeric assay readouts)
=======================================================
                     Titer   HIC   HAC  SEC %Monomer  AC-SINS_pH6.0  Tonset    Tm1    Tm2  mean_abserr
failure_group                                                                                         
consistently_wrong  309.24  2.76  4.01         96.24           0.80   60.48  70.38  81.68       174.15
sometimes_wrong     177.67  2.79  3.90         97.25           0.86   60.82  70.53  83.08       109.68
consistently_right  228.75  2.72  3.75         97.62           0.75   60.89  70.57  82.84        46.25




'''

In [2]:
df = pd.read_csv(r"/home/allen/mads_UMICH/capstone699_repo/data/modeling/failure_eda/cdr_feature_group_stats_246.csv")

In [4]:
dfr = pd.read_csv(r"/home/allen/mads_UMICH/capstone699_repo/data/modeling/failure_eda/gdpa1_with_failure_flags_246.csv")

In [12]:
df.head(63)

,feature,col,p_value,median_diff_wrong_minus_right,median_wrong,median_right
0,cdrL3_len,cdrL3_len_CDR_feature,0.019127,0.000000,9.000000,9.000000
1,cdrL1_charge,cdrL1_charge_CDR_feature,0.022138,0.000000,0.000000,0.000000
2,cdrH1_hyd_frac,cdrH1_hyd_frac_CDR_feature,0.049863,0.090909,0.454545,0.363636
3,cdrL2_oxidn,cdrL2_oxidn_CDR_feature,0.058030,0.000000,0.000000,0.000000
4,cdrL3_isom,cdrL3_isom_CDR_feature,0.111217,0.000000,0.000000,0.000000
5,cdrH3_gravy,cdrH3_gravy_CDR_feature,0.113652,0.167112,-0.555965,-0.723077
6,cdrL3_deamid,cdrL3_deamid_CDR_feature,0.113816,0.000000,0.000000,0.000000
7,cdrL1_len,cdrL1_len_CDR_feature,0.131692,1.500000,10.500000,9.000000
8,cdrL3_charge,cdrL3_charge_CDR_feature,0.132253,0.000000,0.000000,0.000000
9,VL_unpaired_cys,VL_unpaired_cys_CDR_feature,0.136528,0.000000,0.000000,0.000000


In [11]:
dfr.head()

,Unnamed: 0,antibody_id,antibody_name,Titer,Purity,SEC %Monomer,SMAC,HIC,HAC,PR_CHO,PR_Ova,AC-SINS_pH6.0,AC-SINS_pH7.4,Tonset,Tm1,Tm2,hc_subtype,lc_subtype,highest_clinical_trial_asof_feb2025,est_status_asof_feb2025,vh_protein_sequence,hc_protein_sequence,hc_dna_sequence,vl_protein_sequence,lc_protein_sequence,lc_dna_sequence,hierarchical_cluster_fold,random_fold,hierarchical_cluster_IgG_isotype_stratified_fold,light_aligned_aho,heavy_aligned_aho,n_models_high_err,failure_group,mean_abserr,std_abserr,prot_t5_model_abserr,esmc_model_abserr,protbert_model_abserr,ankh_model_abserr,prost_t5_model_abserr,prot_t5_model_pred,esmc_model_pred,protbert_model_pred,ankh_model_pred,prost_t5_model_pred,prot_t5_model_high_err,esmc_model_high_err,protbert_model_high_err,ankh_model_high_err,prost_t5_model_high_err,Unnamed: 0_CDR_feature,cdrH1_len_CDR_feature,cdrH1_gravy_CDR_feature,cdrH1_hyd_frac_CDR_feature,cdrH1_charge_CDR_feature,cdrH1_deamid_CDR_feature,cdrH1_isom_CDR_feature,cdrH1_oxidn_CDR_feature,cdrH2_len_CDR_feature,cdrH2_gravy_CDR_feature,cdrH2_hyd_frac_CDR_feature,cdrH2_charge_CDR_feature,cdrH2_deamid_CDR_feature,cdrH2_isom_CDR_feature,cdrH2_oxidn_CDR_feature,cdrH3_len_CDR_feature,cdrH3_gravy_CDR_feature,cdrH3_hyd_frac_CDR_feature,cdrH3_charge_CDR_feature,cdrH3_deamid_CDR_feature,cdrH3_isom_CDR_feature,cdrH3_oxidn_CDR_feature,cdrL1_len_CDR_feature,cdrL1_gravy_CDR_feature,cdrL1_hyd_frac_CDR_feature,cdrL1_charge_CDR_feature,cdrL1_deamid_CDR_feature,cdrL1_isom_CDR_feature,cdrL1_oxidn_CDR_feature,cdrL2_len_CDR_feature,cdrL2_gravy_CDR_feature,cdrL2_hyd_frac_CDR_feature,cdrL2_charge_CDR_feature,cdrL2_deamid_CDR_feature,cdrL2_isom_CDR_feature,cdrL2_oxidn_CDR_feature,cdrL3_len_CDR_feature,cdrL3_gravy_CDR_feature,cdrL3_hyd_frac_CDR_feature,cdrL3_charge_CDR_feature,cdrL3_deamid_CDR_feature,cdrL3_isom_CDR_feature,cdrL3_oxidn_CDR_feature,cdrH3_aromatic_frac_CDR_feature,cdrH3_glycan_sequon_CDR_feature,VH_deamid_total_CDR_feature,VH_isom_total_CDR_feature,VH_oxidn_total_CDR_feature,VH_glycan_total_CDR_feature,VH_hyd_frac_CDR_feature,VH_gravy_CDR_feature,VH_charge_CDR_feature,VL_deamid_total_CDR_feature,VL_isom_total_CDR_feature,VL_oxidn_total_CDR_feature,VL_glycan_total_CDR_feature,VL_hyd_frac_CDR_feature,VL_gravy_CDR_feature,VL_charge_CDR_feature,total_deamid_CDR_feature,total_oxidn_CDR_feature,VH_unpaired_cys_CDR_feature,VL_unpaired_cys_CDR_feature
0,0,GDPa1-001,abagovomab,140.25,98.530,97.010,2.730,2.590,NaN,0.337837,0.263108,0.35,2.125,62.145,69.535,83.08,IgG1,Kappa,Phase-III,Discontinued,QVKLQESGAELARPGASVKLSCKASGYTFTNYWMQWVKQRPGQGLD...,MRAWIFFLLCLAGRALAQVKLQESGAELARPGASVKLSCKASGYTF...,GCCGCCACCATGAGAGCCTGGATCTTTTTCCTGCTGTGCCTGGCTG...,DIELTQSPASLSASVGETVTITCQASENIYSYLAWHQQKQGKSPQL...,MRAWIFFLLCLAGRALADIELTQSPASLSASVGETVTITCQASENI...,GCCGCCACCATGAGAGCCTGGATCTTTTTCCTGCTGTGCCTGGCTG...,1,2,2,DIELTQSPASLSASVGETVTITCQAS--ENIY------SYLAWHQQ...,QVKLQES-GAELARPGASVKLSCKASG-YTFTN-----YWMQWVKQ...,1.0,sometimes_wrong,88.108057,40.272516,86.800857,56.775752,63.338113,76.547366,157.078199,227.050857,197.025752,203.588113,216.797366,297.328199,0.0,0.0,0.0,0.0,1.0,0,11,-0.763636,0.454545,0,0,0,2,17,-1.188235,0.294118,2,0,1,0,12,-0.633333,0.583333,0,0,0,1,9,-0.122222,0.555556,-1,0,0,0,8,-0.3000,0.500,1,1,0,0,9,-0.622222,0.333333,0,0,0,0,0.333333,0,0,2,7,0,0.386555,-0.502521,4,1,0,1,0,0.355140,-0.257009,1,1,8,0,0
1,1,GDPa1-002,abituzumab,193.31,99.825,97.620,2.745,2.545,3.690,0.205246,0.100155,1.10,1.500,60.630,69.930,80.33,IgG2,Kappa,Phase-II,Active,QVQLQQSGGELAKPGASVKVSCKASGYTFSSFWMHWVRQAPGQGLE...,MRAWIFFLLCLAGRALAQVQLQQSGGELAKPGASVKVSCKASGYTF...,GCCGCCACCATGAGAGCCTGGATCTTTTTCCTGCTGTGCCTGGCTG...,DIQMTQSPSSLSASVGDRVTITCRASQDISNYLAWYQQKPGKAPKL...,MRAWIFFLLCLAGRALADIQMTQSPSSLSASVGDRVTITCRASQDI...,GCCGCCACCATGAGAGCCTGGATCTTTTTCCTGCTGTGCCTGGCTG...,1,4,0,DIQMTQSPSSLSASVGDRVTITCRAS--QDIS------NYLAWYQQ...,QVQLQQS-GGELAKPGASVKVSCKASG-YTFSS-----FWMHWVRQ...,0.0,consistently_right,32.191600,7.806558,33.826759,41.179832,37.205735,21.833185,26.912492,227.136759,234.48983

In [6]:
df.shape

(63, 6)

In [7]:
dfr.shape

(246, 113)